In [ ]:
# Install the core frameworks
!pip install -U langgraph langchain-groq langchain-community langchain-anthropic

# Install SQL helpers and environment management
!pip install sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.8/169.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.8
    Uninstalling langgraph-prebuilt-1.0.8:
      Successfully uninstalled langgraph-prebuilt-1.0.8
  Attemp

In [ ]:
import os
from google.colab import userdata # Use this to hide your key in Colab "Secrets"
from typing import Annotated, List, Union, TypedDict

# LangChain & Groq
from langchain_groq import ChatGroq
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from google.colab import userdata
# LangGraph
from langgraph.graph import StateGraph, END, START

# Set your API Key (Best practice: Put it in Colab's "Secrets" tab on the left)
os.environ["GROQ_API_KEY"] = userdata.get('RealEst')


# Updated to a currently supported, high-performance model
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [ ]:
#getting the sample file

import requests

# Download the sample database
url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
response = requests.get(url)
with open("Chinook.db", "wb") as f:
    f.write(response.content)

print("Chinook.db downloaded successfully!")

Chinook.db downloaded successfully!


In [ ]:
try :
  db = SQLDatabase.from_uri("sqlite:///Chinook.db",)
  print(db.get_usable_table_names())
except Exception as e:
    print(e)



['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


In [ ]:
class AgentState(TypedDict):
  messages:List[str]
  sql_query: str
  db_error: str
  results: str



In [ ]:
def sql_generator(state: AgentState):
    # 1. Get the latest schema
    schema = db.get_table_info()

    # 2. Create a prompt that includes the schema
    prompt = f"Schema: {schema}\nQuestion: {state['messages'][-1]}\nReturn ONLY the SQL query."

    # 3. Get the response
    response = llm.invoke(prompt)

    # 4. Update the state: Add the message to 'messages' AND the query to 'sql_query'
    return {
        "messages": [response],
        "sql_query": response.content
    }



In [ ]:
def db_executor(state: AgentState):
  q= state["sql_query"]
  try:
    res = db.run(q)
    return {"results" :str(res) , "db_error":""}
  except Exception as e:
    return {"results":"", "db_error" : str(e)}





In [ ]:
def should_continue(state : AgentState):
  err = state["db_error"]
  if err :
    return "generate"
  else :
    return "end"


In [ ]:
builder = StateGraph(AgentState)
builder.add_node("generator",sql_generator)
builder.add_node("extractor",db_executor)
builder.add_edge(START , "generator")
builder.add_edge("generator","extractor")
builder.add_conditional_edges(
    "extractor",
    should_continue,
    {
        "generate": "generator",
        "end": END
    }
)

In [ ]:
# 1. Compile the graph
graph = builder.compile()

# 2. Define a tricky question
# (This often trips up models because of the 'InvoiceLine' vs 'Track' join logic)
inputs = {"messages": ["List the top 5 most expensive tracks and their genres."]}

# 3. Run the agent
for output in graph.stream(inputs):
    for key, value in output.items():
        print(f"Node '{key}':")
        # Print a snippet of the state to see what happened
        if "sql_query" in value:
            print(f"  SQL: {value['sql_query']}")
        if "db_error" in value and value["db_error"]:
            print(f"  ERROR FOUND: {value['db_error']}")
        if "results" in value and value["results"]:
            print(f"  RESULTS: {value['results']}")
    print("-" * 20)

Node 'generator':
  SQL: ```sql
SELECT T.Name, G.Name 
FROM Track T 
LEFT JOIN Genre G ON T.GenreId = G.GenreId 
ORDER BY T.UnitPrice DESC 
LIMIT 5;
```
--------------------
Node 'extractor':
  ERROR FOUND: (sqlite3.OperationalError) near "```sql
SELECT T.Name, G.Name 
FROM Track T 
LEFT JOIN Genre G ON T.GenreId = G.GenreId 
ORDER BY T.UnitPrice DESC 
LIMIT 5;
```": syntax error
[SQL: ```sql
SELECT T.Name, G.Name 
FROM Track T 
LEFT JOIN Genre G ON T.GenreId = G.GenreId 
ORDER BY T.UnitPrice DESC 
LIMIT 5;
```]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
--------------------
Node 'generator':
  SQL: ```sql
SELECT T.Name, G.Name 
FROM Track T 
LEFT JOIN Genre G ON T.GenreId = G.GenreId 
ORDER BY T.UnitPrice DESC 
LIMIT 5;
```
--------------------
Node 'extractor':
  ERROR FOUND: (sqlite3.OperationalError) near "```sql
SELECT T.Name, G.Name 
FROM Track T 
LEFT JOIN Genre G ON T.GenreId = G.GenreId 
ORDER BY T.UnitPrice DESC 
LIMIT 5;
```": syntax error
[SQL: ```sql
SELE

KeyboardInterrupt: 

###📦 Project 2: The Multi-Agent Lease Auditor

In [ ]:
# We will also need a PDF parser
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.0 MB/s eta 0:00:00


In [ ]:
class LeaseState(TypedDict):
  lease_text: str
  tenant_concerns:List[str]
  legal_violations : List[str]
  final_report: str



In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def pdf_extractor(state: LeaseState):
    # 1. Provide the actual path to the file you uploaded to Colab
    file_path = "your_lease_file.pdf"
    loader = PyPDFLoader(file_path)

    # 2. Load the pages and join their content into one big string
    pages = loader.load()
    full_text = " ".join([p.page_content for p in pages])

    # 3. Clean up the newlines
    clean_text = full_text.replace("\n", " ")

    return {"lease_text": clean_text}

In [ ]:
def legal_officer(state: LeaseState):
    # 1. Define the "Statutory" Persona
    sys_msg = SystemMessage(content="""You are a Legal Compliance Officer.
    Your goal is to identify clauses that are legally unenforceable or violate housing codes.
    Focus on security deposits, entry rights, and maintenance obligations.
    Provide your findings in a clear, professional list.""")

    # 2. Feed it the lease text
    user_msg = HumanMessage(content=state["lease_text"])

    # 3. Call the LLM
    res = llm.invoke([sys_msg, user_msg])

    # 4. Return as a list to match the TypedDict
    return {"legal_violations": [res.content]}

In [ ]:
def final_auditor(state: LeaseState):
  sys_msg = SystemMessage(content="""
  You are the Senior Lease Auditor. Review the points provided by the Advocate and the Officer. Create a final 'Decision Support'
  summary for the client. Highlight the top 3 risks and give a final 'Sign' or 'Negotiate' recommendation.
  """)
  user_msg = HumanMessage(content = state["tenant_concerns"] + state["legal_violations"])
  res = llm.invoke([sys_msg, user_msg])
  return {"final_report": res.content}


In [ ]:
from langgraph.graph import StateGraph, START, END

# 1. Initialize
builder = StateGraph(LeaseState)

# 2. Add all your nodes
builder.add_node("extractor", pdf_extractor)
builder.add_node("advocate", tenant_advocate)
builder.add_node("officer", legal_officer)
builder.add_node("judge", final_auditor)

# 3. Create the Parallel Flow
builder.add_edge(START, "extractor")

# This "fans out" the work to two nodes at the same time
builder.add_edge("extractor", "advocate")
builder.add_edge("extractor", "officer")

# Both nodes "fan in" to the judge
builder.add_edge("advocate", "judge")
builder.add_edge("officer", "judge")

builder.add_edge("judge", END)

# 4. Compile
audit_app = builder.compile()

###📦 Project 3: The "Memory-Aware" Rental Assistant

In [ ]:
# state
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from typing import Annotated, TypedDict
class ChatState(TypedDict):
  messages : Annotated[list , add_messages]
  user_prefrences : dict

memory = MemorySaver()


In [ ]:

import json

def memory_node(state: ChatState):
    sys_msg = SystemMessage(content="""Extract user preferences (budget, location, pets) from the text.
    Return ONLY a valid JSON dictionary. If nothing is found, return {}.""")

    user_msg = HumanMessage(content=state["messages"][-1].content)
    res = llm.invoke([sys_msg, user_msg])

    # Convert the string response into a real dictionary
    try:
        prefs = json.loads(res.content)
    except:
        prefs = {}

    return {"user_prefrences": prefs}

In [ ]:
def chat_bot(state: ChatState):
    # 1. Personality with memory injection
    sys_msg = SystemMessage(content=f"""You are a helpful Rental Assistant.
    Use these known user preferences to personalize your answer: {state['user_prefrences']}""")

    # 2. Combine the persona with the FULL message history
    all_messages = [sys_msg] + state["messages"]

    # 3. Get response
    res = llm.invoke(all_messages)

    # 4. Return just the object in a list; LangGraph handles the 'add_messages' logic!
    return {"messages": [res]}

In [ ]:
builder = StateGraph(ChatState)
builder.add_node("memory", memory_node)
builder.add_node("chatbot", chat_bot)
builder.add_edge(START, "memory")
builder.add_edge("memory", "chatbot")
app = builder.compile(checkpointer = memory)


In [ ]:
# Create a config with a specific thread_id (like a user session)
config = {"configurable": {"thread_id": "user_42"}}

# Session 1: Tell the bot something
inputs = {"messages": [HumanMessage(content="Hi, I'm looking for a 1-bedroom in London for under £2000.")]}
app.invoke(inputs, config)

# Session 2: Ask a follow-up later (without repeating your budget!)
inputs2 = {"messages": [HumanMessage(content="What are some good neighborhoods for me?")]}
final_output = app.invoke(inputs2, config)

print(final_output["messages"][-1].content)

Considering your budget of £2000 for a 1-bedroom flat in London, here are some neighborhoods that might interest you:

1. **Clapham**: A vibrant and popular area in south London, known for its lively atmosphere, independent shops, and great transport links. You can find 1-bedroom flats in Clapham for around £1800-2000 per month.
2. **Brixton**: A trendy and eclectic neighborhood in south London, famous for its street food, live music venues, and multicultural vibe. 1-bedroom flats in Brixton can range from £1600-1900 per month.
3. **Camden**: A bustling and artistic area in north London, renowned for its markets, live music scene, and alternative culture. You can find 1-bedroom flats in Camden for around £1800-2000 per month.
4. **Islington**: A charming and upscale neighborhood in north London, known for its beautiful parks, independent shops, and vibrant nightlife. 1-bedroom flats in Islington can range from £1800-2000 per month.
5. **Greenwich**: A historic and picturesque area in s

In [ ]:
input3 = {"messages":[HumanMessage(content ="I am looking for neighborhoods with a lot of green spaces and closer to the city.")]}
ot = app.invoke(input3,config)
print(ot["messages"][-1].content)

If you're looking for neighborhoods with plenty of green spaces and proximity to the city, here are some areas that might interest you:

1. **Regent's Park and Camden**: This area offers a mix of green spaces, including Regent's Park, Primrose Hill, and Camden Green. You'll find plenty of 1-bedroom flats in the area, with prices ranging from £1800-2000 per month.
2. **Islington and Highbury**: Islington has several parks, including Highbury Fields, Rosemary Gardens, and Islington Green. The area is also close to the city, with easy access to the West End and financial districts. 1-bedroom flats in Islington can range from £1800-2000 per month.
3. **Greenwich and Blackheath**: Greenwich is a beautiful area with plenty of green spaces, including Greenwich Park, Blackheath, and the Thames River. You'll find 1-bedroom flats in the area, with prices ranging from £1600-1900 per month.
4. **Battersea and Clapham**: Battersea Park is a large green space with a lake, walking trails, and plenty 